# 08 — Claim 1 revised: what must be modelled about the control loop (C1r, C-M)
Arms (each 3 seeds × 40k, evaluated on the same closed-loop production test set):
* **A, controller-aware**: the production simulator (notebook 06 posterior).
* **B, controller-agnostic**: simulator driven by an exogenous coolant flow, no controller model; the
  posterior conditions on the observed Qc through the features.
* **C, open-loop-trained** (the System I form): coolant flow fixed at the nominal value.
* **M, masking**: production simulator, temperature-only features (T_only).

**PRE-REGISTER (user to fix and date before running):**
* B's exogenous flow: level ~ U(10, Qc_max) L/min per window, OU fluctuation 5% / 10 min (draft).
* Metric: per scenario β coverage, |bias|, sd; Claim 1 "controller modelling necessary" holds if B's β
  coverage < 0.80 or its sd > 1.5 × A's on any of Sc1–Sc4.

In [ ]:
import json, sys, time
from dataclasses import replace
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
NB_DIR = Path.cwd().resolve()
assert NB_DIR.name == "notebooks" and NB_DIR.parent.name == "system_III", f"run from system_III/notebooks (cwd {NB_DIR})"
ROOT = NB_DIR.parent
sys.path.insert(0, str(ROOT / "src"))
import importlib, system_iii
from system_iii import physics as ph, simulator as sim, features as fe, data as da, scenarios as scn
from system_iii import sensors as sen, information as info, config as cfg, rules, metrics as wm
for _m in (ph, sim, fe, da, scn, sen, info, cfg, rules, wm):
    importlib.reload(_m)
R, FIGS, DATA = ROOT / "results", ROOT / "figures", ROOT / "data"
for _d in (R, FIGS, DATA):
    _d.mkdir(exist_ok=True)
P = ph.PLANT
T0 = time.time()

In [ ]:
from system_iii import inference as inf
import pickle
PROD = json.loads((R / "06_production.json").read_text())
_m = [pickle.load(open(f, "rb")) for f in PROD["member_files"]]
post = inf.EnsemblePosterior(_m) if PROD["use_ensemble"] else _m[0]
print("production:", PROD["production_budget"], "ensemble" if PROD["use_ensemble"] else "single", PROD["feature_set"])

In [ ]:
test = da.load_npz(DATA / "06_test.npz")
th = sim.prior_draws(120_000, 8100)
arms = {"B_agnostic": dict(control="exogenous", qc_level=np.random.default_rng(8101).uniform(10, P.Qcmax, 120_000)),
        "C_open_loop": dict(control="fixed", qc_level=np.full(120_000, P.Qc0))}
posts = {"A_aware": post}
for arm, kw in arms.items():
    f = DATA / f"08_bank_{arm}.npz"
    if not f.exists():
        da.save_npz(f, da.make_windows(th, 8100, **kw))
    b = da.load_npz(f); ms = []
    for s in range(3):
        g = R / "08" / f"post_{arm}_s{s}.pkl"; g.parent.mkdir(exist_ok=True)
        if g.exists():
            ms.append(pickle.load(open(g, "rb")))
        else:
            sl = slice(s * 40_000, (s + 1) * 40_000)
            p_, _ = inf.train_npe(b["theta"][sl], b["x"][sl], seed=s); pickle.dump(p_, open(g, "wb")); ms.append(p_)
    posts[arm] = inf.EnsemblePosterior(ms)
# masking arm: production bank, T-only features
bank = da.load_npz(DATA / "06_bank_160k.npz"); it = [fe.NAMES["S1m"].index(n) for n in fe.NAMES["T_only"]]
ms = []
for s in range(3):
    g = R / "08" / f"post_M_Tonly_s{s}.pkl"
    if g.exists():
        ms.append(pickle.load(open(g, "rb")))
    else:
        sl = slice(s * 40_000, (s + 1) * 40_000)
        p_, _ = inf.train_npe(bank["theta"][sl], bank["x"][sl][:, it], seed=s); pickle.dump(p_, open(g, "wb")); ms.append(p_)
posts["M_T_only"] = inf.EnsemblePosterior(ms)
rows = []
for arm, p_ in posts.items():
    sub = test if arm != "M_T_only" else dict(test, x_sbc=test["x_sbc"][:, it], x_sc=test["x_sc"][:, it])
    row, sc_rows = inf.evaluate(p_, sub, seed=0)
    rows += [{"arm": arm, **r} for r in sc_rows]
res = pd.DataFrame(rows); res.to_csv(R / "08_claim1.csv", index=False)
display(res.pivot_table(index="scenario", columns="arm", values=["cov90_truth_b", "bias_b", "sd_b", "class_acc"]).round(3))

## Interpretation — to be written after the run
Write toward the storyline (`docs/system_III_plan.md` §1) and update the claims ledger (§2).